In [2]:
import random
import pandas as pd
import string

def generate_cipher_cot_puzzle():
    """Generates a compound string transformation puzzle with a full reasoning trace."""
    
    words = ["RABBIT", "TEAPOT", "HATTER", "QUEEN", "MIRROR", "POCKET", "GARDEN", "ALICE"]
    
    # 1. Randomise the hidden rules for this specific puzzle
    shift = random.randint(1, 5)
    
    def apply_rule(word):
        """Rule: Caesar shift right by 'shift', then lowercase all vowels."""
        result = ""
        for char in word:
            # Shift character
            if char.isalpha():
                base = ord('A') if char.isupper() else ord('a')
                shifted_char = chr((ord(char) - base + shift) % 26 + base)
            else:
                shifted_char = char
                
            # Vowel rule
            if shifted_char.lower() in 'aeiou':
                result += shifted_char.lower()
            else:
                result += shifted_char.upper()
        return result

    # 2. Pick words for the few-shot prompt
    selected_words = random.sample(words, 4)
    target_word = selected_words.pop() # The one the model needs to solve
    
    # 3. Construct the User Prompt
    prompt = "In Alice's Wonderland, secret messages are transformed using a hidden rule.\n"
    prompt += "Here are some examples of the transformation:\n"
    for w in selected_words:
        prompt += f"{w} -> {apply_rule(w)}\n"
    prompt += f"\nNow, determine the output for: {target_word}"
    
    # 4. Construct the Chain-of-Thought (CoT) Trace
    target_output = apply_rule(target_word)
    
    cot = "<think>\n"
    cot += "To solve this, I need to deduce the transformation rule by analyzing the input-output pairs.\n"
    cot += f"Let's look at the first example: {selected_words[0]} -> {apply_rule(selected_words[0])}.\n"
    
    # Explain the shift deduction
    first_char_in = selected_words[0][0]
    first_char_out = apply_rule(selected_words[0])[0]
    cot += f"1. The first letter '{first_char_in}' becomes '{first_char_out}'. "
    cot += f"The alphabetical distance is +{shift}. Let's verify this shift for other consonants.\n"
    
    # Explain the vowel deduction
    cot += "2. Notice the capitalization. Consonants are uppercase, but let's look at the vowels. "
    cot += "Any letter that becomes a vowel (A, E, I, O, U) after the shift is forced to lowercase.\n"
    
    # Apply to target
    cot += f"3. Now, let's apply this compound rule (Shift +{shift}, vowels lowercase) to the target: {target_word}.\n"
    for char in target_word:
        shifted = apply_rule(char)
        cot += f"   - '{char}' shifted by {shift} becomes '{shifted.upper()}'. "
        if shifted.islower():
            cot += f"Since it is a vowel, it becomes lowercase '{shifted}'.\n"
        else:
            cot += f"Since it is a consonant, it remains uppercase '{shifted}'.\n"
            
    cot += "</think>\n"
    
    # 5. Format exactly as the Kaggle metric expects
    # The final output combines the thought process and the LaTeX boxed answer
    full_response = cot + f"\\boxed{{{target_output}}}"
    
    return {
        "prompt": prompt,
        "full_response": full_response
    }

# Generate 5,000 synthetic rows
print("Generating synthetic dataset...")
synthetic_data = [generate_cipher_cot_puzzle() for _ in range(5000)]
synthetic_df = pd.DataFrame(synthetic_data)

# Save to CSV so we can upload it to Kaggle
synthetic_df.to_csv("synthetic_cipher_cot.csv", index=False)
print("Saved to synthetic_cipher_cot.csv")

Generating synthetic dataset...
Saved to synthetic_cipher_cot.csv
